In [1]:
from PIL import Image

import torch
import torch.nn as nn 
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.transforms.v2 import MixUp, CutMix, RandomChoice

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from timm.utils import ModelEmaV3
from timm.data import ImageNetInfo

from CNN import ImageNeuralNetwork

/home/simon/rocm-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Download directly to local Colab disk (fast, ~20-45 seconds)
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip -P /content/

# Unzip locally (fast, no Drive involved)
!unzip -q /content/tiny-imagenet-200.zip -d /content/

# Restructure val/ into class subfolders (train/ is already correctly structured)
import os
import shutil

val_dir = '/content/tiny-imagenet-200/val'
images_dir = os.path.join(val_dir, 'images')
annotations_file = os.path.join(val_dir, 'val_annotations.txt')

with open(annotations_file, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        filename, class_id = parts[0], parts[1]
        class_dir = os.path.join(val_dir, class_id)
        os.makedirs(class_dir, exist_ok=True)
        src = os.path.join(images_dir, filename)
        dst = os.path.join(class_dir, filename)
        if os.path.exists(src):
            shutil.move(src, dst)

shutil.rmtree(images_dir, ignore_errors=True)

In [2]:
best_accuracy = 70.12

def check_accuracy(cnn):
    global best_accuracy 
    correct = 0
    total = 0
    cnn.eval()

    with torch.no_grad(): 
        for data in test_loader:
            images, labels = data
            
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = cnn(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    cnn.train()
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy}%')
    if(accuracy > best_accuracy):
        best_accuracy = accuracy
        torch.save(cnn.state_dict(), f'trained_net_{accuracy}.pth')
    
def load_image(image_path, new_transform):
    image = Image.open(image_path).convert('RGB')
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [4]:
train_transform = transforms.Compose([
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),  
    transforms.RandomErasing(p=0.25),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),
])

In [5]:
train_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/train', transform=train_transform)
test_data = torchvision.datasets.ImageFolder(root='data/tiny-imagenet-200/val', transform=test_transform
)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=512, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=512, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

In [6]:
num_epochs = 300
net = ImageNeuralNetwork(64, 4, 6, 200).to(device)
ema_net = ModelEmaV3(net, decay = 0.999)
loss_function = nn.CrossEntropyLoss(label_smoothing = 0.1)
optimizer = optim.SGD(net.parameters(), lr = 0.1, momentum = 0.9, weight_decay = 1e-4, nesterov = True)
scheduler = CosineAnnealingLR(optimizer, T_max = num_epochs, eta_min = 1e-7)

In [7]:
mixup = MixUp(alpha=0.2, num_classes=200)
cutmix = CutMix(alpha=1.0, num_classes=200)
mixup_cutmix = RandomChoice([mixup, cutmix])

In [ ]:
NUM_NO_MIX = 15
for epoch in range(1, num_epochs + 1):
    
    print(f'Training epoch {epoch}...\n')
    
    running_loss = 0.0
    correct = 0
    total = 0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        if epoch <= num_epochs - NUM_NO_MIX:
            inputs, labels = mixup_cutmix(inputs, labels)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        ema_net.update(net)

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        if labels.dim() == 1:   # only count when NOT mixup/cutmix (hard labels)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    print(f'Loss: {running_loss / len(train_loader):.4f}, LR: {current_lr:.6f}')

    if total > 0:
        train_accuracy = 100 * correct / total
        print(f'Train Accuracy: {train_accuracy:.2f}%\n')
    
    if(epoch >= 140 and epoch % 20 == 0):
        check_accuracy(ema_net.module)

In [ ]:
new_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4802, 0.4481, 0.3975], std=[0.2770, 0.2691, 0.2821]),
])

In [ ]:
image_paths = ['/home/simon/Tiny-ImageNet-Image-Classifier/python-backend/uni.jpg']
images = [load_image(img, new_transform) for img in image_paths]
image_info = ImageNetInfo(subset='imagenet-1k')
net.eval()
with torch.no_grad():
    for image in images:
        outputs = net(image.to(device))
        _, predicted = torch.max(outputs, 1)
        wnid = train_data.classes[predicted.item()]
        print(f'Prediction: {image_info.label_name_to_description(wnid)}')